# Build 04-03 · Does mitigation erase the corruption_footprint? (ShapDiDDelta)

**`notebook/real/mitigation/03_06_shap.ipynb` is now a pointer to this notebook** (fixed
2026-09-12) — it used to be a duplicate stub with the same stated purpose, now redirects here.

**What this measures.** 04_02 gave one number, `corruption_footprint_before` = the v2->v3 DiD on
the UNCORRECTED models. If the mitigation pipeline (`notebook/real/mitigation/03_01`-`03_05`)
actually removes the forced-label contamination, re-running the identical estimator on the
CORRECTED retrain should give a smaller number:

```
ShapDiDDelta = corruption_footprint_before - corruption_footprint_after
```

**§4 (2026-09-13 addition)** repeats this before/after-mitigation comparison a second time,
restricted to a grid-selected band around τ (region_local A'/B') — the same sharp Regression
Discontinuity Design (RDD) `04_02`'s `local_cross_version_estimate` applies to the baseline
version-pair estimate, where `simpson(B') - simpson(A')` at one version's own τ is a boundary
discontinuity in the group-level concentration functional (analogous to a Local Average Treatment
Effect (LATE), not a literal one — see `04_02`'s markdown and `sec:bg-causal`'s
$\Delta\phi_{\mathrm{boundary}}$ construction) — a sharper check on whether the correction
actually moves the claims sitting right on the boundary, not just the whole scrapped population
on average.

**Terminology note.** The corrector `notebook/real/mitigation/03_02_reweight_mitigation.ipynb`
builds (`mitigator.corrector.ReweightCorrector`) is NOT propensity-score IPS — it only borrows the
*reweighting* idea (down-weight/transport unverified labels), not the IPS estimator itself
(`src/mitigator/corrector/reweight.py`'s own docstring says so explicitly). Call it "the reweight
corrector" or "the mitigation pipeline", never "IPS-corrected", in this notebook and in the thesis.

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import schema
import figstyle
import feature_alias
from loaders import load
from estimator import concentration

figstyle.apply()
pd.set_option("display.width", 160)
print("ROOT =", ROOT)

## Prerequisite (not yet done anywhere): SHAP on the MITIGATED models

Nothing computes attributions for a retrained/mitigated model yet. New config kind added
2026-09-12 for exactly this: `config.path("mitigated_attributions", v, source, split=split)` ->
`src/data/{source}/mitigation/shap/{v}_mitigated_attributions_{split}.parquet` (a dedicated kind,
not the baseline "attributions" kind plus a suffix — it belongs with "corrected"/"mitigated"/
"reeval_scores" in the mitigation artefact family). Unlike those three, this kind's own path
carries no axis — §1's `MITIGATION_AXIS` below is what fixes which axis's SHAP is expected there.

Run (in v3's own env, and v2's IF a mitigated v2 model exists) — `00_SHAP.ipynb` (2026-09-14)
detects a mitigated run OFF `MODEL_PATH` itself, never a separate flag: if `MODEL_PATH` points at
one of `03_03_retrain.ipynb`'s per-axis pickles
(`src/models/real/mitigated/<v>_<split>_<axis>.pkl`), it redirects figures/tables into
`figures/mitigated/` AND defaults `OUT_PATH` to
`config.path("mitigated_attributions", VERSION, SOURCE, split=SPLIT)` with `_<axis>` appended —
nothing else needs to be set by hand:

```
MODEL_PATH = ".../src/models/real/mitigated/<v>_<split>_<axis>.pkl"   # the ONLY thing to set;
                                                                       # everything else follows
```

**The axis MODEL_PATH points at here MUST match this notebook's `MITIGATION_AXIS` (§1)** — e.g.
if `00_SHAP.ipynb` was run with `.../v3_train_transport.pkl`, `MITIGATION_AXIS` here must be
`"transport"`, or §3's read of `v3`'s `mitigated_attributions` file will not match what this
notebook's caveats and text assume it is. `mitigated_attributions` has only one file slot per
(version, split) — running `00_SHAP.ipynb` a second time for a different axis overwrites it, so
running two axes side by side means saving the first one's parquet under a different name by hand
before starting the second run. Do the same v3 run for v2 too IF
`notebook/real/mitigation/03_03_retrain.ipynb`'s v2 run actually produced a mitigated model under
`src/models/real/mitigated/v2_<split>_<axis>.pkl` for the SAME axis (v2's own coverage caveats —
the vehicle-status join in 03_01 — mean it may not have, or may only have `naive`/`transport`
since v2 has no score for rarity/pnu); §3 below checks the actual `mitigated_attributions` file
directly and falls back to v2's BASELINE attributions with a printed warning if it's missing,
since v2 may never have been corrected on this axis.</cell id="cell-03">

In [ ]:
# §1 — headline machinery, duplicated (compactly) from 04_02 rather than imported, since these
# are notebooks, not a shared module. Keep this in sync with 04_02 by hand if either changes.
ID_COL = schema.CLAIM_ID
TAG_COLS = [schema.DATE, "score", schema.DECISION, "region", "era", schema.OBSERVED]

# The ONE correction axis this whole notebook compares against baseline. "mitigated_attributions"
# carries no axis in its OWN kind path (unlike "corrected"/"mitigated"/"reeval_scores", which do)
# -- there is exactly one file slot per (version, split), so this constant is what fixes which
# axis's SHAP is expected to sit there. MUST match whatever axis 00_SHAP.ipynb's MODEL_PATH used
# to write that file (see the Prerequisite cell above), and MUST be the same axis for v2 and v3
# (see §3's existence check) -- comparing v2's naive against v3's transport would conflate "which
# correction method" with "how much dose", not measure mitigation at all.
MITIGATION_AXIS = "transport"


def attributions_path(version: str, split: str, mitigated: bool) -> Path:
    if not mitigated:
        return config.path("attributions", version, split=split)
    base = config.path("mitigated_attributions", version, split=split)
    return base.with_name(f"{base.stem}_{MITIGATION_AXIS}{base.suffix}")


def load_cell_table(version: str, split: str, mitigated: bool = False) -> pd.DataFrame:
    """shap_did_input tags (built by 04_01, region/era from the BASELINE decision rule) joined to
    either the baseline or the mitigated model's attributions for the SAME claims.

    Region/era tags don't change under mitigation -- only the SHAP values reasoning about those
    same claims change -- so 04_01's tags are reused unmodified for the "after" arm too.
    """
    tags = pd.read_parquet(config.split_path("shap_did_input", version, split))
    attrs_path = attributions_path(version, split, mitigated)
    if not attrs_path.exists():
        raise FileNotFoundError(
            f"[{version}] {'mitigated' if mitigated else 'baseline'} attributions missing at\n"
            f"    {attrs_path}\nSee the Prerequisite cell above.")
    attrs = pd.read_parquet(attrs_path)
    return tags.merge(attrs, on=ID_COL, how="inner")


def cell_simpson(table: pd.DataFrame, **filters) -> float:
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS if c in sub.columns]
    mabs = concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)
    return concentration.simpson(mabs)


def version_did(version: str, split: str, mitigated: bool = False) -> dict:
    table = load_cell_table(version, split, mitigated)
    s = {(r, e): cell_simpson(table, region=r, era=e) for r in ("A", "B") for e in ("early", "late")}
    did = (s[("B", "late")] - s[("A", "late")]) - (s[("B", "early")] - s[("A", "early")])
    return {"n_claims": len(table), "did": did,
            **{f"simpson_{r}_{e}": v for (r, e), v in s.items()}}

In [ ]:
# §2 — corruption_footprint_before: identical to 04_02's headline (v2 baseline -> v3 baseline)
BEFORE_SPLIT = {"v2": "train", "v3": "train"}   # match 04_02's dose_response row exactly

did_before = {v: version_did(v, BEFORE_SPLIT[v], mitigated=False) for v in ("v2", "v3")}
corruption_footprint_before = did_before["v3"]["did"] - did_before["v2"]["did"]
print("corruption_footprint_before (baseline v2 -> v3):", f"{corruption_footprint_before:+.4f}")
before_table = pd.DataFrame(did_before).T
display(before_table)
# Index is a version, not a feature name -- plain export, no alias twin needed.
figstyle.save_table(before_table, "02_shap_did_mitigation_before")

In [ ]:
# §3 — corruption_footprint_after: v3 MITIGATED, v2 mitigated-if-it-exists-else-baseline (printed
# either way, since silently substituting baseline for a version the user THINKS was corrected
# would misreport what ShapDiDDelta actually measures). Checked against the actual
# mitigated_attributions FILE for MITIGATION_AXIS, not against 03_03's retrained pkl -- a pkl can
# exist without its SHAP ever having been computed, and the pkl carries no axis-match guarantee
# with what THIS notebook expects to read.
v2_mitigated_attrs = attributions_path("v2", BEFORE_SPLIT["v2"], mitigated=True)
V2_MITIGATED = v2_mitigated_attrs.exists()
print(f"v2 mitigated attributions (axis={MITIGATION_AXIS!r}) "
      f"{'FOUND' if V2_MITIGATED else 'NOT FOUND'} at {v2_mitigated_attrs}")
print("  -> v2 arm of corruption_footprint_after uses",
      "the MITIGATED model" if V2_MITIGATED else
      f"the BASELINE model (v2 may never have been corrected on axis={MITIGATION_AXIS!r}, "
      "or its SHAP hasn't been computed yet -- see the Prerequisite cell above)")

did_after = {
    "v2": version_did("v2", BEFORE_SPLIT["v2"], mitigated=V2_MITIGATED),
    "v3": version_did("v3", BEFORE_SPLIT["v3"], mitigated=True),
}
corruption_footprint_after = did_after["v3"]["did"] - did_after["v2"]["did"]
print("\ncorruption_footprint_after ({} v2 -> mitigated v3):".format(
    "mitigated" if V2_MITIGATED else "baseline"), f"{corruption_footprint_after:+.4f}")
after_table = pd.DataFrame(did_after).T
display(after_table)
# Index is a version, not a feature name -- plain export, no alias twin needed.
figstyle.save_table(after_table, "03_shap_did_mitigation_after")

shap_did_delta = corruption_footprint_before - corruption_footprint_after
pct_erased = shap_did_delta / corruption_footprint_before if corruption_footprint_before else float("nan")
print(f"\nShapDiDDelta = {corruption_footprint_before:+.4f} - {corruption_footprint_after:+.4f} "
      f"= {shap_did_delta:+.4f}  (~{pct_erased:.0%} erased)")

## Caveats to print alongside this number, not bury in a footnote

- **Positivity is dead at τ.** Mitigated v3's region B (score > τ) has ZERO garage-verified rows
  by construction (`project_ips_positivity_dead`) — the corrector's effect on region B rests on
  extrapolation (`g(x)` transport, never a verified label), so `corruption_footprint_after` is not
  measured the same way `corruption_footprint_before` is. State this every time the delta is quoted.
- **Not IPS.** Say "reweight corrector" / "the mitigation pipeline", not "IPS-corrected" — see the
  terminology note at the top.
- **v2's own mitigation status must be stated explicitly** — §3 prints which one it used. A
  ShapDiDDelta computed with v2 baseline vs v2 mitigated compares different things even though the
  formula looks identical.
- **Parallel trends caveat carries over from 04_02 unchanged.** Mitigating the training data does
  not touch the v1->v2 era/label-source jump — if 04_02's probe found that violated, this delta
  inherits the same limitation.
- Not yet run against real data — no mitigated attributions exist yet anywhere (see Prerequisite).

## Local RDD boundary discontinuity at τ, before vs after mitigation — does the correction shrink the footprint right at τ?

Everything above compares the WHOLE garage-verified population (region A) against the WHOLE
model-scrapped population (region B), on each side of mitigation. That inherits the same
parallel-trends weak point `04_02`'s headline has (`subsec:shap-did`): A and B do not share a
covariate distribution, so part of `corruption_footprint_before`/`_after` could reflect that
distributional gap rather than the forced-label mechanism the corrector is meant to fix. `04_02`'s
`local_cross_version_estimate` (2026-09-13) addresses this for the BASELINE pair by restricting to
a grid-selected band `|score - tau| <= h` around each row's own τ — a sharp Regression
Discontinuity Design (RDD), where `simpson(B') - simpson(A')` at one version's own τ is a boundary
discontinuity in the group-level concentration functional (analogous to a Local Average Treatment
Effect (LATE), not a literal one — see `04_02`'s markdown: it is built from each side's POOLED
mean-`|phi|` profile, not from an average of each claim's own concentration, the same distinction
`sec:bg-causal`'s $\Delta\phi_{\mathrm{boundary}}$ construction already carries). This section is
that same restriction applied HERE — before vs after mitigation, not before vs after version — so
it answers a sharper question: **does the correction actually move the claims sitting right on
the boundary, or only the ones far from it?**

**h is selected on the BASELINE tables only, then reused unchanged for the mitigated arm.** This
is not just convenience — it is `subsec:shapdiddelta` rule 2 ("the DiD partition must stay
exogenous to the corrector"): if `h` were re-selected on the mitigated table, a scheme that
duplicates rows (the transport soft-split, `subsec:transport`) could shift which band width clears
the power gate, and the analysis would then be comparing two DIFFERENT partitions across arms
instead of the same partition read twice. Same grid, same `MIN_CELL_N`/`H_MAX`/fallback constants
as `04_02` (`project_reweight_corrector_0302`'s procedure, not its number) — re-derived here
because this notebook's population (`train`, per `BEFORE_SPLIT`) differs from `04_02`'s (OOT).

**v1 is out of scope here for the same two reasons it already is for the rest of this notebook**:
`sec:exec-mitigation` states mitigation starts at v2 (v1 has no forced-label $U$ cell to correct),
and its scrap rule has no scalar τ to band around in the first place
(`project_v1_mobility_not_a_feature`).

**The positivity caveat is, if anything, sharper here than for the whole-population number**:
region B' sits by construction in `(tau, tau+h]`, the narrowest possible slice of the exact zone
`corruption_footprint_after`'s caveat already names as having zero garage-verified rows. Read the
local numbers below with that caveat carried over unchanged, not relaxed.

In [ ]:
# §4 — local (boundary-band) ShapDiDDelta: same before/after-mitigation mechanism as SS2/SS3,
# restricted to a grid-selected band around tau (v2->v3 only -- see markdown above)
LOCAL_H_GRID = [0.005, 0.0075, 0.01, 0.015, 0.02, 0.03, 0.05, 0.075, 0.1]   # same grid as 04_02
LOCAL_MIN_CELL_N = 100
LOCAL_H_MAX = 0.05
LOCAL_H_FALLBACK = 0.01


def _tau_by_date(table: pd.DataFrame, version: str) -> pd.Series:
    dates = pd.to_datetime(table[schema.DATE])
    tau_by_date = {d: config.threshold_on(version, d) for d in dates.unique()}
    return dates.map(tau_by_date)


def local_region(table: pd.DataFrame, version: str, h: float) -> pd.DataFrame:
    """region_local A'/B' by distance to that row's OWN tau, band |score - tau| <= h. Strict '>'
    matches the project's region convention: exactly at the band edge above tau is B', not A'."""
    dist = table["score"] - _tau_by_date(table, version)
    band = table[dist.abs() <= h].copy()
    band["region_local"] = np.where(dist[dist.abs() <= h] > 0, "B", "A")
    return band


def cell_simpson_local(table: pd.DataFrame, **filters) -> float:
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS + ["region_local"] if c in sub.columns]
    mabs = concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)
    return concentration.simpson(mabs)


def cell_mabs_local(table: pd.DataFrame, **filters) -> pd.Series:
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS + ["region_local"] if c in sub.columns]
    return concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)


def version_did_local(version: str, split: str, h: float, mitigated: bool = False) -> dict:
    """version_did's twin, restricted to the boundary band -- region_local (A'/B') x era, within
    ONE version, exactly mirroring version_did's region x era DiD but on local_region()'s
    band-filtered table instead of the whole population. Not to be confused with 04_02's
    local_cross_version_estimate, which DiDs a boundary discontinuity ACROSS two versions in one
    call -- this one stays per-version so the before/after-mitigation DiD (computed by the caller,
    same as section §3's whole-population version) uses the identical shape as version_did.
    """
    table = local_region(load_cell_table(version, split, mitigated), version, h)
    s = {(r, e): cell_simpson_local(table, region_local=r, era=e)
         for r in ("A", "B") for e in ("early", "late")}
    did_local = (s[("B", "late")] - s[("A", "late")]) - (s[("B", "early")] - s[("A", "early")])
    return {"n_claims": len(table), "did_local": did_local,
            **{f"simpson_{r}_{e}": v for (r, e), v in s.items()}}


def print_corrector_band_h_reference(version: str, split: str) -> None:
    """Reference only -- NOT used in any computation here. Reads whatever band_h(s) 03_02
    actually selected when it corrected this (version, split) population (rarity/pnu axes only;
    naive/transport carry no band_h and are silently skipped), so the training-time band and this
    notebook's own measurement-time LOCAL_H are always visible side by side, never conflated."""
    base = config.split_path("corrected", version, split)
    metas = sorted(base.parent.glob(f"{base.stem}_*_meta.json"))
    if not metas:
        print(f"[{version}/{split}] no corrected _meta.json found -- band_h reference "
              f"unavailable (run 03_02_reweight_mitigation.ipynb first).")
        return
    printed = False
    for m in metas:
        diag = json.loads(m.read_text(encoding="utf-8"))
        if "band_h" in diag:
            tag = m.stem[len(base.stem) + 1: -len("_meta")]
            print(f"[{version}/{split}] 03_02's own training band_h ({tag}) = {diag['band_h']}")
            printed = True
    if not printed:
        print(f"[{version}/{split}] found {len(metas)} corrected run(s), none carry band_h "
              f"(naive/transport only, or rarity/pnu was skipped -- see 03_02).")


def select_local_h() -> tuple[float, pd.DataFrame]:
    """Selected on the BASELINE v2/v3 train tables ONLY, on cell counts alone -- never on a
    concentration number, and never on a mitigated table (see markdown: rule 2, exogenous
    partition). The same h is reused unchanged for every version_did_local call below.

    This is a MEASUREMENT-time band width, unrelated to 03_02's TRAINING-time `band_h` (see
    print_corrector_band_h_reference() below) -- it is selected independently because band_h only
    exists for the rarity/pnu schemes (naive/transport have none) and governs what the corrector
    up-weighted during retraining, not what this notebook measures afterwards on the fitted SHAP
    values. The two are printed side by side, never forced equal.
    """
    base = {v: load_cell_table(v, BEFORE_SPLIT[v], mitigated=False) for v in ("v2", "v3")}
    dist = {v: t["score"] - _tau_by_date(t, v) for v, t in base.items()}

    rows = []
    for h in sorted(LOCAL_H_GRID):
        counts = {}
        for v, t in base.items():
            in_band = dist[v].abs() <= h
            region_local = np.where(dist[v][in_band] > 0, "B", "A")
            era = t.loc[in_band, "era"].to_numpy()
            for r in ("A", "B"):
                for e in ("early", "late"):
                    counts[f"{v}_{r}_{e}"] = int(((region_local == r) & (era == e)).sum())
        rows.append({"h": h, **counts, "all_pass": min(counts.values()) >= LOCAL_MIN_CELL_N})
    bt = pd.DataFrame(rows).set_index("h")

    ok = bt.index[bt["all_pass"] & (bt.index <= LOCAL_H_MAX)]
    if len(ok):
        chosen = float(ok[0])
        print(f"selected h={chosen} (smallest with every v2/v3 x region_local x era cell >= "
              f"{LOCAL_MIN_CELL_N}, h<={LOCAL_H_MAX})")
    else:
        chosen = LOCAL_H_FALLBACK
        print(f"!! no h <= {LOCAL_H_MAX} reaches {LOCAL_MIN_CELL_N} claims in every cell on "
              f"v2/v3 train -- the near-boundary band is thin at every local width tried. Keeping "
              f"fallback h={LOCAL_H_FALLBACK}; read the local ShapDiDDelta below as LOW-POWER.")
    display(bt)
    return chosen, bt


LOCAL_H, local_h_table = select_local_h()
figstyle.save_table(local_h_table, "04_shap_did_mitigation_local_band_h_selection")
print()
print("reference only, not used above -- 03_02's own training-time band_h for this population:")
print_corrector_band_h_reference("v2", BEFORE_SPLIT["v2"])
print_corrector_band_h_reference("v3", BEFORE_SPLIT["v3"])
print()

did_before_local = {v: version_did_local(v, BEFORE_SPLIT[v], LOCAL_H, mitigated=False) for v in ("v2", "v3")}
corruption_footprint_before_local = did_before_local["v3"]["did_local"] - did_before_local["v2"]["did_local"]
print(f"corruption_footprint_before_local (baseline v2 -> v3, h={LOCAL_H}):",
      f"{corruption_footprint_before_local:+.4f}")
before_local_table = pd.DataFrame(did_before_local).T
display(before_local_table)
figstyle.save_table(before_local_table, "04_shap_did_mitigation_before_local")

did_after_local = {
    "v2": version_did_local("v2", BEFORE_SPLIT["v2"], LOCAL_H, mitigated=V2_MITIGATED),
    "v3": version_did_local("v3", BEFORE_SPLIT["v3"], LOCAL_H, mitigated=True),
}
corruption_footprint_after_local = did_after_local["v3"]["did_local"] - did_after_local["v2"]["did_local"]
print("\ncorruption_footprint_after_local ({} v2 -> mitigated v3, h={}):".format(
    "mitigated" if V2_MITIGATED else "baseline", LOCAL_H), f"{corruption_footprint_after_local:+.4f}")
after_local_table = pd.DataFrame(did_after_local).T
display(after_local_table)
figstyle.save_table(after_local_table, "04_shap_did_mitigation_after_local")

shap_did_delta_local = corruption_footprint_before_local - corruption_footprint_after_local
pct_erased_local = (shap_did_delta_local / corruption_footprint_before_local
                     if corruption_footprint_before_local else float("nan"))
print(f"\nlocal ShapDiDDelta (h={LOCAL_H}) = {corruption_footprint_before_local:+.4f} - "
      f"{corruption_footprint_after_local:+.4f} = {shap_did_delta_local:+.4f} "
      f"(~{pct_erased_local:.0%} erased)")

print(f"\nwhole-population ShapDiDDelta = {shap_did_delta:+.4f}   "
      f"local ShapDiDDelta (h={LOCAL_H}) = {shap_did_delta_local:+.4f}")
print("-> agreement in sign and rough size: the whole-population reading is not just picking up")
print("   a covariate-distribution difference between region A and B.")
print("-> local much smaller, ~0, or opposite sign: the correction's whole-population effect may")
print("   be coming from claims far from the boundary, not from the ones the loop actually forces.")

### Per-feature view: which features carry the near-boundary mass, baseline vs mitigated

The Simpson numbers above are one scalar per (region_local, arm) cell, not which features moved.
Four panels per version — region A' (just below τ) and B' (just above τ), each baseline and
mitigated — real name and alias, same convention as everywhere else in this notebook family. A'
is the control panel here: mitigation targets the forced-label region, so A' is expected to move
little; if it moves as much as B' does, that is itself worth reporting (the corrector may be
doing more than just correcting the forced labels). v2 is only plotted if a mitigated v2 model
exists (§3 above) — baseline vs baseline would show nothing.

In [ ]:
# Feature-level view of the local band, baseline vs mitigated, one version at a time -- v2 and v3
# do not share a feature space, so each version gets its own figure/table (never combined).
LOCAL_CELLS = (("A", "baseline"), ("A", "mitigated"), ("B", "baseline"), ("B", "mitigated"))
LOCAL_TOPK_FEATURES = 10
_LOCAL_CELL_COLOUR = {"A": figstyle.SERIES[0], "B": figstyle.SERIES[1]}


def _local_mitigation_features_fig(mabs_by_cell: dict, version: str, h: float, aliased: bool) -> None:
    fig, axes = plt.subplots(1, len(LOCAL_CELLS), figsize=(4.2 * len(LOCAL_CELLS), 3.6), squeeze=False)
    for ax, (region, arm) in zip(axes[0], LOCAL_CELLS):
        top = mabs_by_cell[(region, arm)].sort_values(ascending=False).head(LOCAL_TOPK_FEATURES)[::-1]
        labels = feature_alias.to_alias(version, top.index) if aliased else list(top.index)
        bars = ax.barh(np.arange(len(top)), top.values, color=_LOCAL_CELL_COLOUR[region])
        ax.set_yticks(np.arange(len(top)))
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f"region {region}' · {arm}")
        ax.set_xlabel("mean |phi|")
        ax.bar_label(bars, fmt="%.3f", fontsize=6, padding=2)
        ax.margins(x=0.12)     # headroom so the value label never clips past the axes edge
    fig.suptitle(f"{version} · local band (h={h}) — feature attribution, baseline vs mitigated"
                 + (" (aliased)" if aliased else ""))
    fig.tight_layout()
    prefix = "alias_" if aliased else ""
    figstyle.save(fig, f"{prefix}{version}_04_shap_did_mitigation_local_band_cell_features")
    plt.show()


_versions_to_plot = ["v3"] + (["v2"] if V2_MITIGATED else [])
if not V2_MITIGATED:
    print("v2 has no mitigated model -- skipping its local band feature-level figure "
          "(baseline vs baseline would show nothing new).")

for _v in _versions_to_plot:
    _table_base = local_region(load_cell_table(_v, BEFORE_SPLIT[_v], mitigated=False), _v, LOCAL_H)
    _table_mit = local_region(load_cell_table(_v, BEFORE_SPLIT[_v], mitigated=True), _v, LOCAL_H)
    _mabs = {
        ("A", "baseline"): cell_mabs_local(_table_base, region_local="A"),
        ("A", "mitigated"): cell_mabs_local(_table_mit, region_local="A"),
        ("B", "baseline"): cell_mabs_local(_table_base, region_local="B"),
        ("B", "mitigated"): cell_mabs_local(_table_mit, region_local="B"),
    }
    _local_mitigation_features_fig(_mabs, _v, LOCAL_H, aliased=False)
    _local_mitigation_features_fig(_mabs, _v, LOCAL_H, aliased=True)

    _wide = pd.DataFrame({f"{r}_{a}": _mabs[(r, a)] for r, a in LOCAL_CELLS})
    _wide.index.name = "feature"
    figstyle.save_table(_wide.sort_values("B_baseline", ascending=False),
                       f"{_v}_04_shap_did_mitigation_local_band_cell_features")
    _alias_wide = _wide.set_axis(feature_alias.to_alias(_v, list(_wide.index)), axis=0).rename_axis("feature")
    figstyle.save_table(_alias_wide.sort_values("B_baseline", ascending=False),
                       f"alias_{_v}_04_shap_did_mitigation_local_band_cell_features")

## Notes

- **§4 local ShapDiDDelta is a sharp Regression Discontinuity Design (RDD), analogous to a Local
  Average Treatment Effect (LATE) at τ, not a literal one** — the mitigation-delta twin of
  `04_02`'s `local_cross_version_estimate` (2026-09-13). `simpson(B') - simpson(A')` inside the
  band `|score - tau| <= h`, for one version at its own τ, is a boundary discontinuity in the
  GROUP-LEVEL concentration functional — exactly the construction the thesis already commits to
  for $\Delta\phi_{\mathrm{boundary}}$ (`eq:boundary-gap`, `sec:bg-causal`): "a SHAP-space
  analogue of $\tau_{\mathrm{SRD}}$ itself ... in place of an outcome". It stops short of a
  literal LATE because `simpson(B') - simpson(A')` is built from each side's POOLED mean-$|\phi|$
  profile (`eq:meanabsshap`, the group-level definition the whole-population `corruption_footprint`
  numbers above already use), not from an average of each claim's OWN concentration — Simpson is
  convex, so pooling first can understate concentration when different claims key off different
  features, even if every individual claim is highly concentrated on its own. Kept group-level
  deliberately, so this stays comparable to the whole-population ShapDiDDelta above, which uses
  the identical definition. `version_did_local` DiDs that boundary discontinuity across era
  (early/late) within a version, and the whole section DiDs it a second time across
  baseline/mitigated. It answers a narrower question than the whole-population ShapDiDDelta
  above: does the correction move the claims sitting right on the boundary, or only ones far from
  it. Both `tau` above/below (`region_local` A'/B') AND before/after mitigation
  (`baseline`/`mitigated`) are checked, and crossed against each other:
  `did_before_local`/`did_after_local` cross `region_local x era x {baseline, mitigated}`, and the
  4-panel figure below crosses `region_local x {baseline, mitigated}` directly on mean|phi| by
  feature.
- **`LOCAL_H` (measurement) is never the same knob as 03_02's `band_h` (training), and is not
  forced to match it.** `band_h` only exists for the rarity/pnu schemes and only bounds the
  GARAGE side of the corrector's own boundary band — `high = s >= (tau - band_h)` has NO upper
  bound at all, so every scrapped row is `high` regardless of how far above τ it scores (cell 2 =
  "every scrapped row + verified band total losses", never grid-searched on its own — there is no
  mirrored scrapped-side band to select). `LOCAL_H` is instead selected independently, on this
  notebook's own (region_local x era) cells, so it applies uniformly across all four schemes
  (naive/transport have no `band_h` at all). `print_corrector_band_h_reference()` prints — for
  reference only, never used in a computation — whatever `band_h` 03_02 actually selected for
  rarity/pnu on this population, read from its `_meta.json` sidecars, so the two widths are always
  visible side by side rather than silently conflated.
- **Reproducibility audit (2026-09-13), across `notebook/real/mitigation/` and
  `notebook/real/shap_did/`.** Everything in THIS notebook (§0-§4) is deterministic — grid
  searches over cell counts, joins, Simpson/mean|phi| arithmetic, no sampling anywhere. Upstream:
  - `03_02_reweight_mitigation.ipynb`'s transport model (`ReweightCorrector._transport`,
    `LogisticRegression(random_state=self.seed)`, default `seed=42`, not overridden by the
    notebook) is seeded.
  - `03_03_retrain.ipynb` / `training.retrain.retrain()`: `model = clone(base)` copies the
    baseline estimator's `random_state`, which matters here because BOTH real baselines have
    stochastic tree-building (`config.TRAINING_CONFIG`: v2 `colsample_bytree=0.6`, v3
    `colsample_bytree=0.887`/`subsample=0.980`) — this is not a moot setting. Both baselines carry
    a fixed `random_state` in `TRAINING_CONFIG` (v2=42, v3=123), and `clone()` propagates whatever
    the LOADED pickle's own `random_state` actually is. `retrain()` was updated (2026-09-13) to
    print a WARNING and record `random_state` in the `_meta.json` sidecar if the loaded baseline's
    own `get_params()["random_state"]` is `None` — so a silent non-reproducibility gap can no
    longer pass unnoticed; check that field once real mitigated models exist.
  - `00_SHAP.ipynb`/`00_SHAP_v1.ipynb` set `SEED = 0` for background/explain-row sampling — this
    notebook's `mitigated_attributions` inherit that, unchanged.
  - Not seed-related, but a real caveat: XGBoost's `n_jobs=-1` (multi-threaded histogram building)
    can produce tiny floating-point differences run-to-run even under a fixed `random_state` —
    a library-level parallelism artefact, not a missing seed, and not fixed here (pinning
    `n_jobs=1` would remove it at a real training-time cost; not recommended by default).
- **Positivity/terminology/etc. caveats**: see the "Caveats" markdown above §2/§3 — they carry
  over unchanged to §4, and §4's own markdown states where the local check's caveat is sharper.